# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [2]:
#!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/25

In [1]:
#!pip uninstall -y numpy pyarrow datasets pandas transformers sentence-transformers huggingface_hub pandas
#!pip install pandas==2.2.2 numpy==1.26.4 "pyarrow==14.0.2" "datasets==2.18.0" transformers sentence-transformers huggingface_hub -U



Found existing installation: numpy 2.4.2
Uninstalling numpy-2.4.2:
  Successfully uninstalled numpy-2.4.2
Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Found existing installation: datasets 4.6.0
Uninstalling datasets-4.6.0:
  Successfully uninstalled datasets-4.6.0
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
  Using cached transformers-5.2.0-py3-none-any.whl.m

In [1]:
import numpy as np
print(np.__version__)

1.26.4


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [93]:
# load the dataset from huggingface in streaming mode and shuffle it
from datasets import load_dataset
wiki_data = load_dataset(
    'vblagoje/wikipedia_snippets_streamed',
    split='train',
    streaming=True
).shuffle(seed=960)

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for vblagoje/wikipedia_snippets_streamed contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/vblagoje/wikipedia_snippets_streamed
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [3]:
# show the contents of a single document in the dataset
#next(iter(wiki_data))

{'wiki_id': 'Q7649565',
 'start_paragraph': 20,
 'start_character': 272,
 'end_paragraph': 24,
 'end_character': 380,
 'article_title': 'Sustainable Agriculture Research and Education',
 'section_title': "2000s & Evaluation of the program's effectiveness",
 'passage_text': "preserving the surrounding prairies. It ran until March 31, 2001.\nIn 2008, SARE celebrated its 20th anniversary. To that date, the program had funded 3,700 projects and was operating with an annual budget of approximately $19 million. Evaluation of the program's effectiveness As of 2008, 64% of farmers who had received SARE grants stated that they had been able to earn increased profits as a result of the funding they received and utilization of sustainable agriculture methods. Additionally, 79% of grantees said that they had experienced a significant improvement in soil quality though the environmentally friendly, sustainable methods that they were"}

In [94]:
# The 'wiki_snippets' dataset does not have 'section_title', so we will proceed without this specific filter
history = iter(wiki_data)

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [95]:
from tqdm.auto import tqdm  # progress bar
from itertools import islice

total_doc_count = 10000 # you can consider 10000 also

counter = 0
docs = []
# iterate through the dataset and apply our filter
for d in tqdm(islice(history, total_doc_count), total=total_doc_count):
    # extract the fields we need - article, section, and passage
    docs.append([d['article_title'], d['section_title'], d['passage_text']])
    # increase the counter on every iteration
    counter += 1


  0%|          | 0/10000 [00:00<?, ?it/s]

In [96]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)

df.columns = ['article_title', 'section_title', 'passage_text']

print(df.shape)

df.head()

(10000, 3)


,article_title,section_title,passage_text
0,Sustainable Agriculture Research and Education,2000s & Evaluation of the program's effectiveness,preserving the surrounding prairies. It ran un...
1,State Street Bank & Trust Co. v. Signature Fin...,Bilski,"a claim is patent-eligible under § 101,"" and ""..."
2,Thames Punting Club,Punt racing,boat that your opponent can. The narrowest of ...
3,The Case of the Late Pig,Plot summary,The Case of the Late Pig Plot summary As Lugg ...
4,Stradivarius (horse),2018: four-year-old season,"place in the straight, caught Torcedor inside ..."


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [ ]:
#!pip uninstall -y pinecone-client pinecone -q
#!pip install "pinecone[grpc]" -U -q  # Latest v5 with gRPC

In [98]:
from google.colab import userdata
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec
from google.colab import userdata

# initialize connection to pinecone (get API key at app.pinecone.io)
#pinecone_api_key = os.environ.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'
pinecone_api_key = userdata.get('PINECONE_API_KEY')

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [99]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [100]:
index_name = "abstractive-question-answering" #give your index a meaningful name

import time

# check if index already exists (it shouldn't if this is first time)
#initialize the index, and insure the stats are all zeros
if(index_name not in pc.list_indexes().names()):
  pc.create_index(name=index_name, dimension=768, metric="cosine", spec=spec)

index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [101]:
import torch
from sentence_transformers import SentenceTransformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = HuggingFaceEmbeddings(model_name="flax-sentence-embeddings/all_datasets_v3_mpnet-base") #load the retriever model from HuggingFace. Use the flax-sentence-embeddings/all_datasets_v3_mpnet-base model
retriever

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: flax-sentence-embeddings/all_datasets_v3_mpnet-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='flax-sentence-embeddings/all_datasets_v3_mpnet-base', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [102]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from uuid import uuid4

# we will use batches of 64
batch_size = 64

#['article_title', 'section_title', 'passage_text']

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
)

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):
    batch_texts = []
    metas = []

    # get end of batch
    i_end = min(len(df), i+batch_size)

    # extract batch
    batch = df.iloc[i:i_end]

    # find end of batch
    for _, row in batch.iterrows():
        chunks = text_splitter.split_text(row['passage_text'])

        for chunk in chunks:
          batch_texts.append(chunk)

          # create metadata for each row of the dataframe
          metas.append({
              "passage_text": chunk,
              "article_title": row['article_title'],
              "section_title": row['section_title']
          })

    # create unique IDs
    ids = [str(uuid4()) for _ in range(len(batch_texts))]

    # generate embeddings for batch
    embeddings = retriever.embed_documents(batch_texts)

    index.upsert(vectors=list(zip(ids, embeddings, metas)))

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/157 [00:00<?, ?it/s]

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 22123}},
 'total_vector_count': 22123,
 'vector_type': 'dense'}

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [103]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [105]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.embed_documents([query])
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [106]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = "".join(context)
    # contcatinate the query and context passages
    query = f"question: {query}context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [107]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

{'matches': [{'id': 'eb32582c-1d00-4f61-835f-744abd831e58',
              'metadata': {'article_title': 'ULTra (rapid transit)',
                           'passage_text': 'As part of the development of the '
                                           'first commercial system',
                           'section_title': 'Ultra'},
              'score': 0.571003914,
              'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

In [108]:
from pprint import pprint

In [109]:
# format the query in the form generator expects the input
query = format_query(query, result['matches'])
pprint(query)

('question: when was the first electric power system built?context: <P> As '
 'part of the development of the first commercial system')


The output looks great. Now let's write a function to generate answers.

In [110]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [111]:
generate_answer(query)

('The first electric power system was probably the first steam engine. The '
 'first steam engine was probably the first steam engine. The first steam '
 'engine was probably the first steam engine. The first steam engine was')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [112]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by Morse code, which was a form of Morse '
 'code that was used to communicate with other people. The first wireless '
 'message was sent by radio, which was a form')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [113]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

to broadcast a digital signal in 2001.
---
may have difficulty communicating when there is a strong 800 MHz broadband transmitter nearby.
---
It is unknown whether the model 20B receiver had a beat frequency oscillator that would enable the detection of continuous wave transmissions such as Morse code and radiolocation beacons. Neither Earhart nor Noonan were capable of using Morse code. They relied on voice communications. Manning, who was on the first world flight attempt but not the second, was skilled at Morse and had acquired an
---
Earhart's 7:58 am transmission said she couldn't hear the Itasca and asked them to send voice signals so she could try to take a radio bearing. This transmission was reported by the Itasca as the loudest possible signal, indicating Earhart and Noonan were in the immediate area. They couldn't send voice at the frequency she asked for, so Morse code signals were sent instead. Earhart acknowledged
---
and 6210 kHz. The Electra had been equipped to transmi

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [114]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 was first identified in the US in the late 90s. It was first '
 'identified in the US in the early 2000s. It was first identified in the US '
 'in the early')


In [115]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

lightning from dry thunderstorms on June 20, although some earlier fires ignited during mid-May. International aid from Greece, Cyprus, Chile, Argentina, Brazil, Australia, Canada, Mexico, and New Zealand helped fight the fires.
---
about and try to investigate it for himself. While he was there, he collected hundreds of gallons of samples of blood and urine and sweat, trying to find how it spread (and it
---
pathogens and prepared for deployment.
---
in Kenya and India so far.
---
India.
---


Let’s finish with a final few questions.

In [116]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The War of Current was a dispute between Westinghouse and Edison over the '
 'use of alternating current (AC) in electrical equipment. Westinghouse wanted '
 'to use direct current, and Edison wanted to')


In [118]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The first man to walk on the moon was Neil Armstrong. He walked on the moon '
 'in 1969. He landed on the far side of the Moon in 1969.')


In [119]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of NASA. It '
 'cost about $10 billion to build.')


In [92]:
#pc.delete_index(index_name)

As we can see, the model can generate some decent answers.

#### Add a few more questions

In [121]:
query = "What is the radius of Earth?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The radius of the Earth is about 3.5 million km. The radius of the Moon is '
 'about 1.5 million km. The radius of the Sun is about 1.5 million km.')


In [122]:
query = "How many planets are in our solar system?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The total number of planets in our solar system is estimated to be about 18 '
 'quintillion. The total number of stars in our solar system is estimated to '
 'be about 4.5 billion.')
